## Exploratory Data Analysis 

### Imports and loading data 

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

os.makedirs('artifacts', exist_ok=True)

In [ ]:
df = pd.read_csv('Wildfire_Dataset.csv', parse_dates=['datetime'])

print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
print('\nDtypes:\n', df.dtypes)
df.head()

### Looking at the data structure

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())
print('\nBasic statistics:')
df.describe()

In [ ]:
rows_per_location = df.groupby(['latitude', 'longitude']).size()
print('Rows per location (multiples of 75 expected):')
print(rows_per_location.value_counts().head(10))

### checking the sentinel value - note GRIDMET encodes missing readings as 32767.0

In [ ]:
FILL_VALUE = 32767.0
FEATURES = ['pr', 'rmax', 'rmin', 'sph', 'srad', 'tmmn', 'tmmx',
            'vs', 'bi', 'fm100', 'fm1000', 'erc', 'etr', 'pet', 'vpd']

sentinel_rows = (df[FEATURES] == FILL_VALUE).any(axis=1).sum()
print(f'Rows containing sentinel value {FILL_VALUE}: {sentinel_rows:,}')
print(f'That is {sentinel_rows / len(df) * 100:.2f}% of all rows')
print('\nSentinel counts per feature:')
print((df[FEATURES] == FILL_VALUE).sum())

# looking into class balance 

In [ ]:
sample_rows = []
for (lat, lon), loc_df in df.groupby(['latitude', 'longitude']):
    loc_df = loc_df.sort_values('datetime').reset_index(drop=True)
    for start in range(0, len(loc_df) - 75 + 1, 75):
        window = loc_df.iloc[start:start + 75]
        if len(window) == 75:
            row = window.iloc[0].copy()
            row['Wildfire'] = 'Yes' if (window['Wildfire'] == 'Yes').any() else 'No'
            sample_rows.append(row)

samples = pd.DataFrame(sample_rows).reset_index(drop=True)
print('Total samples:', len(samples))

class_counts = samples['Wildfire'].value_counts()
print('\nSamples per class:')
print(class_counts)
print(f'\nClass balance: {class_counts["Yes"] / len(samples) * 100:.1f}% wildfire, '
      f'{class_counts["No"] / len(samples) * 100:.1f}% non-wildfire')

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(
    class_counts.index,
    class_counts.values,
    color=['#d62728', '#1f77b4'],
    edgecolor='black',
    linewidth=0.8
)
ax.set_title('Class Balance (Samples)', fontsize=13)
ax.set_ylabel('Number of Samples')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for i, (label, count) in enumerate(class_counts.items()):
    ax.text(i, count + 200, f'{count:,}', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('artifacts/class_balance.png', dpi=150)
plt.show()

### setting up the feautures

In [ ]:
FEATURE_LABELS = {
    'pr':     'Precipitation',
    'rmax':   'Max Relative Humidity',
    'rmin':   'Min Relative Humidity',
    'sph':    'Specific Humidity',
    'srad':   'Solar Radiation',
    'tmmn':   'Min Temperature (K)',
    'tmmx':   'Max Temperature (K)',
    'vs':     'Wind Speed',
    'bi':     'Burning Index',
    'fm100':  'Fuel Moisture 100hr',
    'fm1000': 'Fuel Moisture 1000hr',
    'erc':    'Energy Release Component',
    'etr':    'Reference ET',
    'pet':    'Potential Evapotranspiration',
    'vpd':    'Vapor Pressure Deficit'
}

### looking at the distribution of features

In [ ]:
df_plot = df.loc[~(df[FEATURES] == FILL_VALUE).any(axis=1)].copy()

fire = df_plot[df_plot['Wildfire'] == 'Yes']
no_fire = df_plot[df_plot['Wildfire'] == 'No']

fig, axes = plt.subplots(5, 3, figsize=(15, 18))
axes = axes.flatten()

for i, feat in enumerate(FEATURES):
    ax = axes[i]
    ax.hist(no_fire[feat].dropna(), bins=50, alpha=0.6,
            color='#1f77b4', label='No Wildfire', density=True)
    ax.hist(fire[feat].dropna(), bins=50, alpha=0.6,
            color='#d62728', label='Wildfire', density=True)
    ax.set_title(FEATURE_LABELS[feat], fontsize=10)
    ax.set_xlabel(feat)
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.suptitle('Feature Distributions: Wildfire vs Non-Wildfire\n(sentinel values excluded for visualization)',
             fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('artifacts/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

### Temporal Patterns Over the 75-Day Window

In [ ]:
df_plot = df_plot.sort_values(['latitude', 'longitude', 'datetime'])
df_plot['day_idx'] = df_plot.groupby(['latitude', 'longitude']).cumcount() % 75

In [ ]:
temporal_fire = df_plot[df_plot['Wildfire'] == 'Yes'].groupby('day_idx')[FEATURES].mean()
temporal_no_fire = df_plot[df_plot['Wildfire'] == 'No'].groupby('day_idx')[FEATURES].mean()

TEMPORAL_FEATURES = ['vpd', 'erc', 'fm100', 'rmin', 'tmmx', 'vs']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, feat in enumerate(TEMPORAL_FEATURES):
    ax = axes[i]
    ax.plot(temporal_no_fire.index, temporal_no_fire[feat],
            color='#1f77b4', label='No Wildfire', linewidth=1.5)
    ax.plot(temporal_fire.index, temporal_fire[feat],
            color='#d62728', label='Wildfire', linewidth=1.5)
    ax.axvline(x=60, color='gray', linestyle='--', linewidth=1,
               label='Ignition day (approx.)')
    ax.set_title(FEATURE_LABELS[feat], fontsize=10)
    ax.set_xlabel('Day in 75-day window')
    ax.legend(fontsize=8)

plt.suptitle('Mean Feature Values Over 75-Day Window by Class', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('artifacts/temporal_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

### plotting/looking at the geographical distribution

In [ ]:
fire_samples = samples[samples['Wildfire'] == 'Yes']
no_fire_samples = samples[samples['Wildfire'] == 'No']

fig, ax = plt.subplots(figsize=(12, 6))
ax.scatter(
    no_fire_samples['longitude'], no_fire_samples['latitude'],
    s=1, alpha=0.3, color='#1f77b4', label='No Wildfire'
)
ax.scatter(
    fire_samples['longitude'], fire_samples['latitude'],
    s=1, alpha=0.5, color='#d62728', label='Wildfire'
)
ax.set_title('Geographic Distribution of Samples (US)', fontsize=13)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.legend(markerscale=5, fontsize=10)
ax.set_xlim(-130, -60)
ax.set_ylim(24, 50)
plt.tight_layout()
plt.savefig('artifacts/geographic_distribution.png', dpi=150)
plt.show()

### looking at the fire ignitions by the year chart 

In [ ]:
fire_samples = samples[samples['Wildfire'] == 'Yes'].copy()
fire_samples['year'] = pd.to_datetime(fire_samples['datetime']).dt.year
ignitions_by_year = fire_samples.groupby('year').size()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(ignitions_by_year.index, ignitions_by_year.values,
       color='#d62728', edgecolor='black', linewidth=0.7)
ax.set_title('Wildfire Ignition Samples by Year', fontsize=13)
ax.set_xlabel('Year')
ax.set_ylabel('Number of Samples')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('artifacts/ignitions_by_year.png', dpi=150)
plt.show()

### making a correlation matrix of the features

In [ ]:
corr = df_plot[FEATURES].corr()

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

labels = [FEATURE_LABELS[f] for f in FEATURES]
ax.set_xticks(range(len(FEATURES)))
ax.set_yticks(range(len(FEATURES)))
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(labels, fontsize=9)

for i in range(len(FEATURES)):
    for j in range(len(FEATURES)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}',
                ha='center', va='center', fontsize=6)

ax.set_title('Feature Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.savefig('artifacts/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()